# 02 — Propensity score model

**Goal.** Estimate P(treated | covariates) for each pool, and check whether the two groups
overlap enough for matching to be possible at all.

| | |
|---|---|
| **Reads** | `outputs/data/01_sample_cps.csv`, `01_sample_psid.csv` |
| **Writes** | `outputs/data/02_scored_cps.csv`, `02_scored_psid.csv` |

The two pools run as two parallel tracks and are never combined. CPS offers 15,992 candidate
controls, PSID only 2,490, and PSID respondents sit further from the treated group on almost
every covariate. Keeping them separate is what makes the difficulty difference visible.

**Questions this notebook answers**

1. How far apart are the two groups before any adjustment? This is the imbalance the propensity
   model exists to correct, so the baseline belongs here rather than in the data-prep notebook,
   where it cannot be compared against anything. Save the figure as `psm_covariate_gap`.
2. Which covariate specification goes into the model, and why does that choice matter so much?
3. Do the estimated scores overlap between arms, or do the tails have no counterparts?
4. Does a flexible model (gradient boosting) change the picture versus logistic regression?

For the imbalance baseline, hold the denominator fixed with `psm.reference_sd(X, t)` and pass it
to `psm.smd` for both pools. The 185 treated rows are identical across the two samples, so a fixed
denominator is what makes the CPS and PSID columns comparable. Letting each pool supply its own
pooled SD makes them look equally imbalanced on 1975 earnings when the PSID gap is in fact 45%
larger.

A high AUC is not the goal. The propensity score is a balancing tool, not a prediction task:
near-perfect separation would mean the two groups are not comparable at all, which is a finding
about the data rather than a success of the model.

Save every figure with `figures.save_fig(fig, "...")` so it can be reused in the README and on
the portfolio page later.

In [ ]:
import sys
sys.path.insert(0, "..")

# Reload src/ modules automatically whenever they change on disk. Without this,
# Python caches the module on first import and later edits to src/*.py are
# invisible until you restart the kernel.
%load_ext autoreload
%autoreload 2

import json
import warnings

warnings.filterwarnings("ignore", message=".*numexpr.*")
warnings.filterwarnings("ignore", message=".*bottleneck.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import psm, data, figures

pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

POOLS = ["cps", "psid"]

%matplotlib inline

In [ ]:
samples = {pool: data.load_stage(f"01_sample_{pool}") for pool in POOLS}

for pool, df in samples.items():
    print(f"{pool:5s} {df.shape[0]:>6,} rows   treated {int(df.treat.sum()):>3d}"
          f"   treated share {df.treat.mean():.1%}")